# PyTorch 与神经网络模型实践

## 1. PyTorch 基础

### 1.1 张量运算

**请使用 PyTorch 内置函数完成，避免显式循环。**

1. 创建一个形状为 (6, 5) 的随机整数张量 `A`，取值范围为 {0, 1, ..., 10}。打印出该张量；
2. 将 `A` 的每一行减去该行的均值（行中心化），得到张量 `A_centered`，使用广播实现。打印出该张量；
3. 打印出 `A_centered` 的行和向量以及列和向量。

In [ ]:
import torch
A = torch.randint(0, 11, (6, 5))
print("A =")
print(A)
row_mean = A.mean(dim=1, keepdim=True)
A_centered = A - row_mean
print("\nA_centered =")
print(A_centered)
row_sums = A_centered.sum(dim=1)
col_sums = A_centered.sum(dim=0)
print("\nA_centered 的行和向量 =")
print(row_sums)
print("\nA_centered 的列和向量 =")
print(col_sums)

以下每一步都需打印出操作的结果。

1. 创建一个形状为 (8, 8) 的随机正态分布张量 `X`（均值为 0.5，方差为2）；
2. 提取 `X` 中所有大于1.5的元素，构成一维张量；
3. 将 `X` 中绝对值小于0.5的元素替换为0，结果保存为张量 `X_masked`；
4. 取出 `X` 的第2行、第4列的元素；
5. 取出 `X` 的第3-6行、第2-5列构成的子矩阵。

In [ ]:
import torch
X = torch.normal(mean=0.5, std=(2 ** 0.5), size=(8, 8))
print("X =")
print(X)
greater_than_15 = X[X > 1.5]
print("\nX 中大于 1.5 的元素 =")
print(greater_than_15)
X_masked = X.clone()
X_masked[X_masked.abs() < 0.5] = 0
print("\nX_masked =")
print(X_masked)
elem = X[1, 3]
print("\nX 的第2行第4列元素 =")
print(elem)
submatrix = X[2:6, 1:5]
print("\nX 的第3-6行、第2-5列子矩阵 =")
print(submatrix)

### 1.2 常见函数与自动微分

多分类问题的数据通常包括数据阵 $X$ 和标签向量 $l$，其中标签为整数。在计算损失函数时，我们需要先将 $l$ 转换成多项分布的0-1数据，即所谓 One-hot 编码。运行并观察下面的代码。

In [ ]:
import numpy as np
import torch
import torch.nn as nn

np.random.seed(123456)
torch.manual_seed(123456)

n = 200  # 样本量
p = 10   # 变量数
k = 4    # 类别数
x = torch.randn(n, p)
l = torch.tensor(np.random.choice(range(4), size=n, replace=True), dtype=int)
print(l[:20])

y = torch.nn.functional.one_hot(l)
print(y.shape)
print(y[:10])

请创建矩阵 `W`，大小为 $k \times p$，用 N(0, 2) 填充其取值。

In [ ]:
W = torch.normal(mean=0.0, std=(2.0 ** 0.5), size=(k, p))

# 检查 W 的形状，方便 debug
assert W.shape == (k, p), "W 形状有误"

接下来计算对 $Y$ 的概率预测值，其中每个 $Y_i$ 观测对应一个等长的概率向量 $p_i$，而 $p_i=\mathrm{Softmax}(Wx_i)$。首先计算 $Wx_i$，其中 $x_i$ 是第 $i$ 个观测。由于 $X$ 是把 $x_i$ 按行组合，因此矩阵形式表达为 $U=XW'$，其中 $U$ 的第 $i$ 行即为 $Wx_i$。

In [ ]:
U = x @ W.t()

# 检查 U 的形状，方便 debug
assert U.shape == (n, k), "U 形状有误"

我们先测试一下 $\mathrm{Softmax}(Wx_{100})$ 的结果，观察其元素和是否为1。代码中的 `dim=0` 意思是对第一个下标方向计算 Softmax，由于 `U[99]` 是一个向量，因此第一个下标方向就是该向量本身。

In [ ]:
torch.softmax(U[99], dim=0)

而为了对 $U$ 的每一行都计算 Softmax，我们可以直接对整个 `U` 矩阵用 `torch.softmax`，其中 `dim` 需指定为1，意思是对第二个下标方向求 Softmax，即对 $U$ 的每一行。原理类似于按坐标轴汇总。请完成该计算，得到矩阵 $P$，其中 $P$ 的第 $i$ 行即为 $p_i$。

In [ ]:
P = torch.softmax(U, dim=1)

# 检查 P 的形状，方便 debug
assert P.shape == (n, k), "P 形状有误"

根据 `y` 和 `P` 两个矩阵，即可根据公式得到对数似然函数值。总结上述步骤，编写损失函数 `loss_fn_softmax(w, x, y)`，返回**负**对数似然值。

In [ ]:
def loss_fn_softmax(w, x, y):
    U = x @ w.t()
    P = torch.softmax(U, dim=1)
    eps = 1e-12
    loglik = (y * torch.log(P + eps)).sum(dim=1).sum()
    return -loglik

Pytorch 中也提供了 CrossEntropyLoss 损失函数，参见[其文档](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)。其用法是先建立一个损失函数对象，然后将 $U$ 和 $l$ 作为参数传入（注意 $U$ 是经过 Softmax **之前**的矩阵，$l$ 是**原始**的标签）。请利用这种方法计算损失函数值，并与你自己的函数结果进行对比。

In [ ]:
ce_softmax = nn.CrossEntropyLoss()
loss1 = ce_softmax(U, l)

loss2 = loss_fn_softmax(W, x, y)

print(loss1)
print(loss2)

【可以在此处添加必要的说明文字】

利用 PyTorch 的自动微分功能，计算上述损失函数在 $W=O$ 处的梯度，其中 $O$ 是一个元素全为0的矩阵。

In [ ]:
W0 = torch.zeros((k, p), requires_grad=True)
loss0 = loss_fn_softmax(W0, x, y)
loss0.backward()
print("\nW=O 时的梯度：")
print(W0.grad)

## 2. 前馈神经网络

### 2.1 线性模型

利用模块化编程（参考课件 `lec6-module.ipynb` 中的实现），在如下模拟数据上构建一个 Logistic 回归模型（包含截距项），并利用自动微分和优化器求解回归系数。要求使用 PyTorch 中的 `nn.Linear` 模块完成模型构建。

In [ ]:
import numpy as np

def gen_data(n_obs=1000, radius=1.0, eye_oversample_ratio=0.0):
    factor = 1.5
    n_candidates = int(n_obs * factor)
    points = np.random.uniform(-radius, radius, size=(n_candidates, 2))

    r_sq = np.sum(points**2, axis=1)
    inside = (r_sq <= radius**2)
    points = points[inside]

    while len(points) < n_obs:
        extra = np.random.uniform(-radius, radius, size=(n_obs, 2))
        inside_extra = (np.sum(extra**2, axis=1) <= radius**2)
        extra = extra[inside_extra]
        points = np.vstack([points, extra])
    points = points[:n_obs]

    x1, x2 = points[:, 0], points[:, 1]
    half_r = radius / 2.0
    eye_r = radius / 5.0

    upper_semi = (x1 > 0) & ((x1**2 + (x2 - half_r)**2) <= half_r**2)
    lower_semi = (x1 < 0) & ((x1**2 + (x2 + half_r)**2) <= half_r**2)

    yang_eye = (x1**2 + (x2 - half_r)**2) <= eye_r**2
    yin_eye = (x1**2 + (x2 + half_r)**2) <= eye_r**2

    labels = np.full(len(points), -1, dtype=int)
    labels[upper_semi] = 0
    labels[yang_eye] = 1
    labels[lower_semi] = 1
    labels[yin_eye] = 0

    rest = (labels == -1)
    labels[rest & (x1 >= 0)] = 1
    labels[rest & (x1 < 0)] = 0

    if eye_oversample_ratio > 0:
        n_extra = int(n_obs * eye_oversample_ratio)

        def sample_disk(center, radius_disk, n):
            pts = np.random.uniform(-radius_disk, radius_disk, size=(int(n * 1.5), 2))
            inside_d = np.sum(pts**2, axis=1) <= radius_disk**2
            pts = pts[inside_d][:n]
            while len(pts) < n:
                extra_pts = np.random.uniform(-radius_disk, radius_disk, size=(n, 2))
                in_extra = np.sum(extra_pts**2, axis=1) <= radius_disk**2
                extra_pts = extra_pts[in_extra]
                pts = np.vstack([pts, extra_pts])[:n]
            return pts + np.array(center)

        extra_yang_pts = sample_disk(center=(0, half_r), radius_disk=eye_r, n=n_extra)
        extra_yang_labels = np.ones(n_extra, dtype=int)

        extra_yin_pts = sample_disk(center=(0, -half_r), radius_disk=eye_r, n=n_extra)
        extra_yin_labels = np.zeros(n_extra, dtype=int)

        points = np.vstack([points, extra_yang_pts, extra_yin_pts])
        labels = np.concatenate([labels, extra_yang_labels, extra_yin_labels])

    return points, labels

np.random.seed(123456)
x, y = gen_data(n_obs=1000, radius=10.0, eye_oversample_ratio=0.1)
print(x[:5])
print(y[:5])

数据散点图：

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

dat = pd.DataFrame(x, columns=["x1", "x2"])
dat["y"] = y
plt.figure(figsize=(6, 6))
sns.scatterplot(data=dat, x="x1", y="x2", hue="y")

请在下方完成训练代码：

In [ ]:
import torch
import torch.nn as nn
W = torch.zeros((k, p), requires_grad=True)
optimizer = torch.optim.SGD([W], lr=0.1)
num_epochs = 200
loss_history = []

for epoch in range(num_epochs):
    optimizer.zero_grad()
    loss = loss_fn_softmax(W, x, y)
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Loss = {loss.item():.4f}")
print("\n训练完成后的 W =")
print(W.detach())
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.grid(True)
plt.show()


完成模型训练后，利用得到的模型对如下测试集数据进行预测（概率 >0.5 判为1，反之判为0），计算分类的正确率。

In [ ]:
np.random.seed(654321)
xtest, ytest = gen_data(n_obs=200, radius=10.0, eye_oversample_ratio=0.1)

In [ ]:
import torch
xtest_t = torch.tensor(xtest, dtype=torch.float32)
ytest_t = torch.tensor(ytest, dtype=torch.long)
U_test = xtest_t @ W.t()
P_test = torch.softmax(U_test, dim=1)
y_pred = (P_test[:, 1] > 0.5).long()
accuracy = (y_pred == ytest_t).float().mean().item()
print("测试集正确率 =", accuracy)

### 2.2 前馈神经网络

修改上面的线性模型，将其变为一个两层的前馈神经网络，隐藏神经元数量为32，使用 ReLU 激活函数。然后重新训练模型（可尝试使用不同的学习率和迭代次数），记录每次迭代的损失函数值并绘制损失函数值随迭代次数的曲线。最后对测试集进行预测，计算分类的正确率（目标是 >90%）。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
x_train = torch.tensor(x, dtype=torch.float32)
y_train = torch.tensor(y, dtype=torch.long)
x_test = torch.tensor(xtest, dtype=torch.float32)
y_test = torch.tensor(ytest, dtype=torch.long)
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=2, hidden_dim=32, output_dim=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 2000
loss_history = []

for epoch in range(num_epochs):
    model.train()
    
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}/{num_epochs}, Loss = {loss.item():.6f}")

plt.figure(figsize=(8, 5))
plt.plot(loss_history, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.grid(True)
plt.show()
model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    y_pred = torch.argmax(test_logits, dim=1)
    acc = (y_pred == y_test).float().mean().item()
print(f"测试集分类正确率 = {acc * 100:.2f}%")


### 2.3 Muon 优化器

请重新对上述前馈神经网络进行初始化和训练，但使用 Muon 优化器。**注意：Muon 只能对矩阵型参数进行优化，把向量型的参数传递给 Muon 会报错。**

请使用如下思路解决：构建两个优化器，Adam 和 Muon，将模型的参数分成两个列表，矩阵类的传递给 Muon，其他类的传递给 Adam，然后在训练循环中分别调用两个优化器的 `zero_grad()` 和 `step()` 函数。

请记录下这种方式下每次迭代的损失函数值，并与上一节中的曲线绘制在同一张图中进行对比。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
x_train = torch.tensor(x, dtype=torch.float32)
y_train = torch.tensor(y, dtype=torch.long)

x_test = torch.tensor(xtest, dtype=torch.float32)
y_test = torch.tensor(ytest, dtype=torch.long)
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, output_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

model = MLP(input_dim=2, hidden_dim=32, output_dim=2)

muon_params = []
adam_params = []

for name, param in model.named_parameters():
    if param.ndim >= 2:
        muon_params.append(param)  
    else:
        adam_params.append(param)   

print("Muon 参数：", [p.shape for p in muon_params])
print("Adam 参数：", [p.shape for p in adam_params])

criterion = nn.CrossEntropyLoss()
optimizer_muon = Muon(muon_params, lr=0.01)
optimizer_adam = torch.optim.Adam(adam_params, lr=0.01)
num_epochs = 2000
loss_history_muon = []

for epoch in range(num_epochs):
    model.train()
    
    optimizer_muon.zero_grad()
    optimizer_adam.zero_grad()
    
    logits = model(x_train)
    loss = criterion(logits, y_train)
    
    loss.backward()
    
    optimizer_muon.step()
    optimizer_adam.step()
    
    loss_history_muon.append(loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}/{num_epochs}, Loss = {loss.item():.6f}")

plt.figure(figsize=(8, 5))

try:
    plt.plot(loss_history, label="Adam only", linewidth=2)
except NameError:
    print("未找到上一节的 loss_history，请先运行上一节训练并保存 loss_history。")

plt.plot(loss_history_muon, label="Muon + Adam", linewidth=2)

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.grid(True)
plt.legend()
plt.show()

model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    y_pred = torch.argmax(test_logits, dim=1)
    acc = (y_pred == y_test).float().mean().item()

print(f"测试集分类正确率 = {acc * 100:.2f}%")


## 3. 预训练模型实战

请学习语音识别模型 Whisper 的使用方法（访问 [https://hf-mirror.com/openai/whisper-small](https://hf-mirror.com/openai/whisper-small) 或 [https://huggingface.co/openai/whisper-small](https://huggingface.co/openai/whisper-small)），完成以下任务：

1. 准备一段文字（内容任意，朗读出来大约10秒钟），填写在下方的文本框里。
2. 朗读这段文字，利用录音软件录制一段10秒左右的音频，保存为 `.wav` 格式。
3. 根据网站上的文档学习使用 Whisper 预训练模型，将预训练模型下载到本地，并识别你录制的音频，将其转为文字，打印出来。
4. 配备有 Nvidia GPU 的同学可以尝试参数量更大的 Whisper-Large 模型（访问 [https://hf-mirror.com/openai/whisper-large-v3](https://hf-mirror.com/openai/whisper-large-v3) 或 [https://huggingface.co/openai/whisper-large-v3](https://huggingface.co/openai/whisper-large-v3)），在 CUDA 模式下运行模型。

真实文字内容：
顺成人，逆成仙，玄妙只在颠倒间。
躲天意、避因果，诸般枷锁困真我。
顺天意、承因果，今日方知我是我。
语音文件的文件名：C:\Users\19716\xwechat_files\wxid_rf7j0otk4fs422_628e\msg\file\2026-05\20260525_101145.m4a

In [ ]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(
    r"C:\Users\19716\xwechat_files\wxid_rf7j0otk4fs422_628e\msg\file\2026-05\20260525_101145.m4a",
    language="zh",
    task="transcribe"
)
print(result["text"])

结果：检测到的语言：zh，置信度：0.98

转录文本：
[0.00s -> 5.12s] 顺成人，逆成仙，玄妙只在颠倒间。
[5.12s -> 10.58s] 躲天意、避因果，诸般枷锁困真我。
[10.58s -> 16.24s] 顺天意、承因果，今日方知我是我。